<a href="https://colab.research.google.com/github/jtmuraski/nfl-analytics/blob/main/Nfl_Vikings_ByeWeek_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set up


In [1]:
## Install needed libraries
!pip install nflreadpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.2 MB/s eta 0:00:00


In [2]:
import nflreadpy as nfl
import pandas as pd
import seaborn as sb
import os
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

print("All libraries have been successfully imported")
print("Pandas version: ", pd.__version__)

All libraries have been successfully imported
Pandas version:  2.2.2


# Schedule Data

Here we import the schedule data, filter for Vikings games and then do some cleanup, exploration and add the necessary columns for calculating bye weeks, and post bye week game results

In [3]:
schedules = nfl.load_schedules(True)
print(schedules.head())

shape: (5, 46)
┌─────────────┬────────┬───────────┬──────┬───┬─────────────┬────────────┬────────────┬────────────┐
│ game_id     ┆ season ┆ game_type ┆ week ┆ … ┆ home_coach  ┆ referee    ┆ stadium_id ┆ stadium    │
│ ---         ┆ ---    ┆ ---       ┆ ---  ┆   ┆ ---         ┆ ---        ┆ ---        ┆ ---        │
│ str         ┆ i32    ┆ str       ┆ i32  ┆   ┆ str         ┆ str        ┆ str        ┆ str        │
╞═════════════╪════════╪═══════════╪══════╪═══╪═════════════╪════════════╪════════════╪════════════╡
│ 1999_01_MIN ┆ 1999   ┆ REG       ┆ 1    ┆ … ┆ Dan Reeves  ┆ Gerry      ┆ ATL00      ┆ Georgia    │
│ _ATL        ┆        ┆           ┆      ┆   ┆             ┆ Austin     ┆            ┆ Dome       │
│ 1999_01_KC_ ┆ 1999   ┆ REG       ┆ 1    ┆ … ┆ Dick Jauron ┆ Phil       ┆ CHI98      ┆ Soldier    │
│ CHI         ┆        ┆           ┆      ┆   ┆             ┆ Luckett    ┆            ┆ Field      │
│ 1999_01_PIT ┆ 1999   ┆ REG       ┆ 1    ┆ … ┆ Chris       ┆ Bob        ┆ C

In [4]:
print(schedules.columns)

['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [5]:
# Convert to pandas DataFrame explicitly
# This is done to make the df usable in the Colabds Data Inspector
schedules_pd = schedules.to_pandas()
pd.set_option('display.max_columns', None)

In [6]:
# Filter out the schedule to get just the Vikigns schedule
neededCols = [
    "game_id",
    "season",
    "game_type",
    "week",
    "gameday",
    "away_team",
    "home_team",
    "away_score",
    "home_score",
    "result",
    "away_rest",
    "home_rest",
    "away_coach",
    "home_coach",
]

vikingsScheduleFullPd = schedules_pd[
    (
        (schedules_pd["home_team"] == "MIN")
        | (schedules_pd["away_team"] == "MIN")
    )
    & (schedules_pd["game_type"] == "REG")
]
vikingsSchedule = vikingsScheduleFullPd[neededCols].copy()

# Add in the points differential for each Vikings game > 0, vikings win, == 0 tie and < 0 == loss
vikingsSchedule["differential"] = np.where(
    vikingsSchedule["home_team"] == "MIN",
    vikingsSchedule["home_score"] - vikingsSchedule["away_score"],
    vikingsSchedule["away_score"] - vikingsSchedule["home_score"],
)

# Remove rows where 'differential' is NaN (i.e., future games)
vikingsSchedule.dropna(subset=['differential'], inplace=True)

# Add column for Vikings results
vikingsSchedule["vikings_win"] = np.where(vikingsSchedule["differential"] > 0, 1, 0)
vikingsSchedule["vikings_loss"] = np.where(vikingsSchedule["differential"] < 0, 1, 0)
vikingsSchedule["vikings_tie"] = np.where(vikingsSchedule["differential"] == 0, 1, 0)

# Determine if game is post bye or not
restDaysForBye = 13
vikingsSchedule["had_bye_week"] = np.where(
    (
        (vikingsSchedule["home_team"] == "MIN")
        & (vikingsSchedule["home_rest"] >= restDaysForBye)
    )
    |
    (
        (vikingsSchedule["away_team"] == "MIN")
        & (vikingsSchedule["away_rest"] >= restDaysForBye)
    ),
    1,
    0,
)

# Perform a sanity check and list the number of bye weeks for each season
# with the exception of 2001 beacuse of 9/11, each season should only have 1 bye
print(vikingsSchedule.groupby(["season"])["had_bye_week"].sum())
print("Total post bye games: " + str(vikingsSchedule["had_bye_week"].sum()))


season
1999    1
2000    1
2001    2
2002    1
2003    1
2004    1
2005    1
2006    1
2007    1
2008    1
2009    1
2010    1
2011    1
2012    1
2013    1
2014    1
2015    1
2016    1
2017    1
2018    1
2019    1
2020    1
2021    1
2022    1
2023    1
2024    1
2025    1
Name: had_bye_week, dtype: int64
Total post bye games: 28


#Begin Analysis
Now that the schedule has been filtered for Vikings games and we now know which games are bye week games, and the scores - we can see how the Vikings fair in post bye week contests

In [15]:
import pandas as pd
import numpy as np

# Initialize an empty list to store the results for each category
analysis_results = []

# --- Total Games Analysis ---
total_games_count = len(vikingsSchedule)
total_wins = vikingsSchedule['vikings_win'].sum()
total_losses = vikingsSchedule['vikings_loss'].sum()
total_ties = vikingsSchedule['vikings_tie'].sum()

analysis_results.append({
    'Category': 'Total Games',
    'Games Played': total_games_count,
    'Wins': total_wins,
    'Losses': total_losses,
    'Ties': total_ties,
    'Win %': round((total_wins / total_games_count) * 100 if total_games_count > 0 else 0 ,2),
    'Loss %': round((total_losses / total_games_count) * 100 if total_games_count > 0 else 0, 2),
    'Tie %': round((total_ties / total_games_count) * 100 if total_games_count > 0 else 0, 2)
})

# --- Post-Bye Games Analysis ---
post_bye_games = vikingsSchedule[vikingsSchedule['had_bye_week'] == 1].copy() # Added .copy() here
post_bye_games_count = len(post_bye_games)
post_bye_wins = post_bye_games['vikings_win'].sum()
post_bye_losses = post_bye_games['vikings_loss'].sum()
post_bye_ties = post_bye_games['vikings_tie'].sum()

analysis_results.append({
    'Category': 'Post-Bye Games',
    'Games Played': post_bye_games_count,
    'Wins': post_bye_wins,
    'Losses': post_bye_losses,
    'Ties': post_bye_ties,
    'Win %': round((post_bye_wins / post_bye_games_count) * 100 if post_bye_games_count > 0 else 0, 2),
    'Loss %': round((post_bye_losses / post_bye_games_count) * 100 if post_bye_games_count > 0 else 0, 2),
    'Tie %': round((post_bye_ties / post_bye_games_count) * 100 if post_bye_games_count > 0 else 0, 2)
})

# --- Non-Post-Bye Games Analysis ---
non_post_bye_games = vikingsSchedule[vikingsSchedule['had_bye_week'] == 0].copy() # Added .copy() here
non_post_bye_games_count = len(non_post_bye_games)
non_post_bye_wins = non_post_bye_games['vikings_win'].sum()
non_post_bye_losses = non_post_bye_games['vikings_loss'].sum()
non_post_bye_ties = non_post_bye_games['vikings_tie'].sum()

analysis_results.append({
    'Category': 'Non-Post-Bye Games',
    'Games Played': non_post_bye_games_count,
    'Wins': non_post_bye_wins,
    'Losses': non_post_bye_losses,
    'Ties': non_post_bye_ties,
    'Win %': round((non_post_bye_wins / non_post_bye_games_count) * 100 if non_post_bye_games_count > 0 else 0, 2),
    'Loss %': round((non_post_bye_losses / non_post_bye_games_count) * 100 if non_post_bye_games_count > 0 else 0, 2),
    'Tie %': round((non_post_bye_ties / non_post_bye_games_count) * 100 if non_post_bye_games_count > 0 else 0,2)
})

# Convert the list of results to a DataFrame
game_performance_df = pd.DataFrame(analysis_results)

display(game_performance_df)

,Category,Games Played,Wins,Losses,Ties,Win %,Loss %,Tie %
0,Total Games,437,233,202,2,53.32,46.22,0.46
1,Post-Bye Games,28,14,14,0,50.00,50.00,0.00
2,Non-Post-Bye Games,409,219,188,2,53.55,45.97,0.49


In [11]:
import pandas as pd

# Get the Post Bye split between Home and Away Games
post_bye_home_games_df = post_bye_games[post_bye_games["home_team"] == "MIN"]
post_bye_home_games_count = len(post_bye_home_games_df)
post_bye_home_differential = post_bye_home_games_df["differential"].sum()
post_bye_home_wins = post_bye_home_games_df["vikings_win"].sum()


post_bye_away_games_df = post_bye_games[post_bye_games["away_team"] == "MIN"]
post_bye_away_games_count = len(post_bye_away_games_df)
post_bye_away_differential = post_bye_away_games_df["differential"].sum()
post_bye_away_wins = post_bye_away_games_df["vikings_win"].sum()

post_bye_splits = []
post_bye_splits.append({
    "Category" : "Home Games",
    "# Games" : post_bye_home_games_count,
    "Differential" : post_bye_home_differential,
    "Win %" : round((post_bye_home_wins / post_bye_home_games_count) * 100 if post_bye_home_games_count > 0 else 0, 2)
})

post_bye_splits.append({
    "Category" : "Away Games",
    "# Games" : post_bye_away_games_count,
    "Differential" : post_bye_away_differential,
    "Win %" : round((post_bye_away_wins / post_bye_away_games_count) * 100 if post_bye_away_games_count > 0 else 0, 2)
})

# Convert the list of results to a DataFrame and display
post_bye_splits_df = pd.DataFrame(post_bye_splits)
display(post_bye_splits_df)

,Category,# Games,Differential,Win %
0,Home Games,11,24.0,63.64
1,Away Games,17,-108.0,41.18


In [ ]:
# Get the number of games played and wins grouped by coaches

In [16]:
# Identify the Vikings coach for each game
post_bye_games['vikings_coach'] = np.where(
    post_bye_games['home_team'] == 'MIN',
    post_bye_games['home_coach'],
    post_bye_games['away_coach']
)

# Group by coach and calculate games played and wins
coach_performance = post_bye_games.groupby('vikings_coach').agg(
    games_played=('game_id', 'count'),
    wins=('vikings_win', 'sum')
).reset_index()

# Calculate win percentage
coach_performance['win_percentage'] = (
    (coach_performance['wins'] / coach_performance['games_played']) * 100
).round(2)

# Display the results
display(coach_performance.sort_values(by='games_played', ascending=False))

,vikings_coach,games_played,wins,win_percentage
5,Mike Zimmer,8,3,37.5
0,Brad Childress,5,4,80.0
1,Dennis Green,4,2,50.0
2,Kevin O'Connell,4,2,50.0
4,Mike Tice,4,3,75.0
3,Leslie Frazier,3,0,0.0
